#### OutPut Parser
- The primary purpose of an Output Parser is to take unstructured text responses from a language model and convert them into structured data types, such as dictionaries, lists, or custom objects.
- Every Output Parser object has two main function:
    - `parse`: This function processes the unstructured text response from the language model and returns it as structured data (e.g., JSON, dictionary, etc.).
    - `get_format_instructions()` : This function provides guidelines or instructions for the LLM (Language Model) to format its responses. These instructions define how the output should be structured, and the LLM generates its output accordingly, adhering to the specified format.
- Some popular OutputParser examples are :
    - `JsonOutputParser`: This parser converts the output into a JSON format.
    - `XmlOutputParser`: This parser converts the output into an XML format.
    - `DictOutputParser`: This parser converts the output into a dictionary format.
    - `ListOutputParser`: This parser converts the output into a list format.
    - `CommaSeparatedListOutputParser` : This parser takes a string of comma-separated values and converts it into a list format, useful for processing simple, delimited text.
    - `StrOutputParser` : This parser returns the output as a plain string
- Some most frequently used parser examplea as given below:
    - `StructuredOutputParser`: A flexible parser used when outputs need to adhere to a specific structure, often paired with schema definitions (e.g., pydantic models).
    - `RegexOutputParser`: Parses outputs based on a regular expression pattern. Useful for extracting specific data points from text.
    - `BooleanOutputParser`: Converts the response into a boolean value, typically based on specific keywords or conditions in the response.
    - `NumericOutputParser`: Extracts and returns numeric values from the output, such as integers or floats.

In [1]:
import os
from dotenv import find_dotenv, load_dotenv
_=load_dotenv(find_dotenv())
# openai_api_key = os.environ["OPENAI_API_KEY"]
google_api_key=os.environ["GOOGLE_API_KEY"]
cerebras_api_key=os.environ["CEREBRAS_API_KEY"]
groq_api_key=os.environ["GROQ_API_KEY"] 

##### Checking the functionality of CommaSeparatedListOutputParser

In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
# Create the object of OutputParser
output_parser = CommaSeparatedListOutputParser()
# In the below example, we are parsing the output of the model which is comma separated strings into lists of strings.
# Here , parse method has been used.
##########<<parser function example of Output parser>>################
reply = "one, two, three" # Lets assume this is the output from the model.
print(output_parser.parse(reply))  ##  Opuput will be like : ['one', 'two', 'three']
print("\n\n\n++++++++++++++++++++++++++++++++++++++++++++++++++++++++\n\n\n") 
##########<<Getting the standard formate instructions attached with 'CommaSeparatedListOutputParser'>>################
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

['one', 'two', 'three']



++++++++++++++++++++++++++++++++++++++++++++++++++++++++



Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [ ]:
#### Simple But Complete Example:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=google_api_key)
output_parser = CommaSeparatedListOutputParser()
human_template = '{request} {format_instructions}'
human_prompt = HumanMessagePromptTemplate.from_template(human_template)
chat_prompt = ChatPromptTemplate.from_messages([human_prompt])
request = chat_prompt.format_prompt(request="give me 5 characteristics of dogs",
          format_instructions = output_parser.get_format_instructions())
result = model.invoke(request)
print(result.text)
output_parser.parse(result.text) 

In [ ]:
# CommaSeparatedListOutputParser
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
import os
google_api_key = os.getenv('GOOGLE_API_KEY')
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=google_api_key)
    
def call_list_output_parser():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Generate a list of 10 synonyms for the following word. Return the results as a comma seperated list."),
        ("human", "{input}")
    ])

    parser = CommaSeparatedListOutputParser()
    
    chain = prompt | model | parser

    return chain.invoke({
        "input": "happy"
    }) 
print(call_list_output_parser())

In [ ]:
#### StringOutput Parser 
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
google_api_key = os.getenv('GOOGLE_API_KEY')
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=google_api_key)
def call_string_output_parser():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Tell me a joke about the following subject"),
        ("human", "{input}")
    ])

    parser = StrOutputParser()

    chain = prompt | model | parser

    return (chain.invoke({
    # print("chain value:",chain)
        "input": "dog"
    }))
call_string_output_parser()
# print(type(call_string_output_parser()))

##### Why StringOutput Parser is required , when result.text can give the result in string.
- consider flow  Topic --LLM---blog---LLM--- 5 point summary.

In [ ]:
# Cannot create the chain as result.text component not supported the runnable chain concept
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv,find_dotenv
from langchain_core.prompts import PromptTemplate
_=load_dotenv(find_dotenv())
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",  # "TinyLlama/TinyLlama-1.1B-Chat-v1.0", 
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)
# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)
# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. write 5 lines as bullet points \n {text}',
    input_variables=['text']
)
prompt1 = template1.invoke({'topic':'black hole'})
result = model.invoke(prompt1)
prompt2 = template2.invoke({'text':result.text})
result1 = model.invoke(prompt2)

print(result1.text)

##### Hugging Face → LangChain Model Check (Quick Guide)

1. **Ctrl+F → `text-generation`**
   - Must be present  
   - If missing → ❌ not an LLM

2. **Ctrl+F → `Inference` / `Inference Provider`**
   - Must show deployed inference support  
   - If you see *“not deployed”* → ❌ won’t work in LangChain

3. **Ctrl+F → `GGUF`**
   - If found → ❌ local-only model (llama.cpp / Ollama)

**Rule:**  
`text-generation` + deployed inference − `GGUF` = ✅ works with `HuggingFaceEndpoint`


In [3]:
# Connecting the differnt component all together using runnable chain concept
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv,find_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
_=load_dotenv(find_dotenv())
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct", #"Qwen/Qwen3-1.7B",#google/gemma-2-2b-it",  # "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)
# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)
# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. write 5 lines as bullet points \n {text}',
    input_variables=['text']
)

# Creating the object of StrOutputParser
parser = StrOutputParser()

chain=template1 | model | parser | template2 | model | parser
result=chain.invoke({'topic':'black hole'})

In [4]:
print(result) 

- Black holes are cosmic entities characterized by extremely strong gravity preventing even light from escaping.
- Theoretical understanding of black holes emerged from Einstein's General Relativity and contributions by Schwarzschild, Oppenheimer, and others.
- Black holes are categorized into stellar, intermediate-mass, and supermassive types based on their mass and rotation.
- Stellar black holes form from the collapse of massive stars and typically have masses between 3 to 100 solar masses.
- Intermediate-mass black holes fill a gap in the black hole mass spectrum, linking stellar and supermassive black holes.


#### JSON OUTPUT PARSER
- Force the llm generate the output as json 

In [ ]:
#### Demonstration of Partial variable with Output Parser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()


### In LangChain, input variables are dynamic values supplied at runtime, while partial variables are 
# predefined values injected into the prompt at creation time and remain constant across executions.

template = PromptTemplate(
    template='Give me the name,age and city of a fictional person \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)
### Use this as pair
prompt=template.format() # return string type
print(type(prompt))
print(prompt)
########. OR ########

#### Use this as pair
# prompt=template.format_prompt()  ## return Prompt Type
# print(type(prompt))
# print(prompt.text)


In [ ]:
## JsonOutput Parser with Partial variable example - Complete
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",# repo_id="google/gemma-2-2b-it",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

parser = JsonOutputParser()


### In LangChain, input variables are dynamic values supplied at runtime, while partial variables are predefined values injected into the prompt at creation time and remain constant across executions.

template = PromptTemplate(
    template='Give me the name,age and city of a fictional person \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

# prompt=template.format() # return string type
# print(type(prompt))
# print(prompt)
########. OR ########

prompt=template.format_prompt()  ## return Prompt Type
# print(type(prompt))
# print(prompt.text)

result=model.invoke(prompt)
print('Before parse:',result.text) 
final_result=parser.parse(result.text)
print('After parse:\n', final_result)

In [ ]:
## JsonOutput Parser with Partial variable example - With Chain
## if no input variable still at runtime we have to pass empty dict, whenver we calling the chain.

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",# repo_id="google/gemma-2-2b-it",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

parser = JsonOutputParser()


### In LangChain, input variables are dynamic values supplied at runtime, while partial variables are predefined values injected into the prompt at creation time and remain constant across executions.

template = PromptTemplate(
    template='Give me the name,age and city of a fictional person \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

# prompt=template.format() # return string type
# print(type(prompt))
# print(prompt)
########. OR ########

prompt=template.format_prompt()  ## return Prompt Type
# print(type(prompt))
# print(prompt.text)

chain=template | model | parser
result=chain.invoke({})  ## passing empty dict as no input variable
print(result) 
 


In [ ]:
## With JsonOutput Parser, you can get the json object , but you cant not force the user defined schema/structure .
# i.e Biggest drwback of Jsonparser that you can't enforce the user defined schema/structure .
import json
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it", # repo_id="Qwen/Qwen2.5-Coder-32B-Instruct"
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

parser = JsonOutputParser()


### In LangChain, input variables are dynamic values supplied at runtime, while partial variables are predefined values injected into the prompt at creation time and remain constant across executions.

template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result) 
print(type(result))
print(json.dumps(result, indent=2))

##### JsonOutputParser – Drawback and Remedy (Latest Trend)

#### Drawback
- `JsonOutputParser` ensures the output is valid JSON
  - Returns Python objects (`dict` or `list`)
- ❌ Cannot enforce a user-defined schema or structure
  - Required fields are not guaranteed
  - Field names may change
  - Data types may be inconsistent

---

#### Remedy (Recommended in LangChain 1.x)
- Use **structured output at model level**
  - Enforce structure **during generation**, not after
- Use:
  - JSON Schema, or
  - Pydantic models
- Implemented using:
  - `model.with_structured_output(schema)`

---

##### Important Note
- `StructuredOutputParser` + `ResponseSchema`
  - Legacy approach
  - Not recommended in latest LangChain versions
  - it was not having Data validation.



In [ ]:
###. Old legacy approch that outdated now, lagecy code for demonstrate ###
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
# from dotenv import load_dotenv
# from langchain_core.prompts import PromptTemplate
# from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# load_dotenv()

# # Define the model
# llm = HuggingFaceEndpoint(
#     repo_id="google/gemma-2-2b-it",
#     task="text-generation"
# )

# model = ChatHuggingFace(llm=llm)

# schema = [
#     ResponseSchema(name='fact_1', description='Fact 1 about the topic'),
#     ResponseSchema(name='fact_2', description='Fact 2 about the topic'),
#     ResponseSchema(name='fact_3', description='Fact 3 about the topic'),
# ]

# parser = StructuredOutputParser.from_response_schemas(schema)

# template = PromptTemplate(
#     template='Give 3 fact about {topic} \n {format_instruction}',
#     input_variables=['topic'],
#     partial_variables={'format_instruction':parser.get_format_instructions()}
# )

# chain = template | model | parser

# result = chain.invoke({'topic':'black hole'})

# print(result)

##### JsonOutputParser – Key Points (For Understanding)
- `JsonOutputParser` returns the LLM output as a Python object
  - Output may be a `dict` or a `list`
  - ❌ No data validation : Incorrect data types may pass through
  - ❌ No guarantee of schema or structure
    - Missing fields
    - Inconsistent keys
---
##### Recommended Approach (Best Practice)
- Use **Pydantic models** for structured output
  - Enforces schema
  - Validates data types
  - Suitable for production use
- If needed, convert validated Pydantic output to JSON later
  - Safe and reliable for downstream systems
---
##### One-line Takeaway
> Use `JsonOutputParser` for learning and quick experiments,  
> but use **Pydantic-based structured output** for reliable production systems.

##### What is PydanticOutputParser in LangChain?

- **PydanticOutputParser** is a structured output parser in LangChain that uses **Pydantic models** to enforce **schema validation** when processing LLM responses.
---
##### Why Use PydanticOutputParser?
- ✅ **Strict Schema Enforcement**  
  Ensures that LLM responses follow a well-defined structure.
- ✅ **Type Safety**  
  Automatically converts LLM outputs into validated Python objects.
- ✅ **Easy Validation**  
  Uses Pydantic’s built-in validation to catch incorrect or missing data.
- ✅ **Seamless Integration**  
  Works well with other LangChain components.

In [ ]:
## PydanticOutput Parser with Partial variable example - Complete
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",# repo_id="google/gemma-2-2b-it",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

prompt= template.format_prompt(place='Russia')  ## return Prompt Type
print(prompt.text) 

print("\n\n================================\n\n")
result = model.invoke(prompt)
final_result = parser.parse(result.text)
print(final_result)

In [ ]:
## PydanticOutput Parser with chain
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

final_result = chain.invoke({'place':'sri lankan'})

print(final_result)

### Table 1: Approach Comparison (Structured Output in LangChain)

| Aspect | PydanticOutputParser (Explicit) | with_structured_output() (Implicit) |
|------|----------------------------------|-------------------------------------|
| Prompt control | High (manual instructions) | Low (auto-generated) |
| Parsing control | Manual (`parse()`) | Automatic |
| Debug visibility | High (see raw output) | Low |
| Boilerplate code | More | Less |
| Teaching suitability | Excellent | Poor |
| Error handling | Explicit | Implicit |
| HF model reliability | High | Medium |
| Best use case | Learning, debugging, HF models | Rapid prototyping, strong models |


### Table 2: Model Choice vs Approach Rigidity

| Model Type | Instruction Following | Recommended Approach | Reason |
|-----------|----------------------|----------------------|--------|
| OpenAI GPT-4 / Gemini | Very Strong | with_structured_output() | Models strictly follow format |
| Claude 3.x | Very Strong | with_structured_output() | High JSON discipline |
| Large HF Instruct (e.g. Qwen, Mistral) | Strong | PydanticOutputParser | More prompt control needed |
| HF Code Models | Medium | PydanticOutputParser | Often add explanations |
| Small / Mid HF Models | Weak–Medium | PydanticOutputParser | Require rigid guidance |
| Mixed-quality HF models | Variable | PydanticOutputParser | Safer, debuggable |

---
**Rule:** Better the model → less rigidity needed; weaker the model → more explicit parsing required.


##### Interview Question

Both `with_structured_output` and `JsonOutputParser` in LangChain use a Pydantic schema to produce structured data.  
What is the difference between these two approaches, and which one should be preferred in modern applications?

---

##### Answer

Both approaches share a common foundation in that they use a **Pydantic schema** to define the expected structure of the output. The key difference lies in **when and how** that schema is applied.

`with_structured_output` applies the schema **before generation**, constraining the model at the **protocol level** using capabilities such as JSON mode or function calling. This forces the model to generate output that already conforms to the schema, resulting in **lower failure rates, simpler prompts, stronger guarantees, better agent integration, and a future-proof design**.

On the other hand, `JsonOutputParser` applies the schema **after generation**. The model first produces free-form text, and LangChain then attempts to parse and validate that text against the schema. This approach relies on **prompt obedience**, making it more **fragile**, increasing prompt complexity, and leading to higher chances of parsing errors.

Therefore, in modern applications where models natively support structured output, **`with_structured_output` is the preferred approach**, while `JsonOutputParser` should be used primarily as a fallback for models that do not support schema-level enforcement.